In [ ]:
# Import libraries
import sys
sys.path.insert(0, "..")

import lightgbm                       as lgb
import matplotlib.pyplot              as plt
import pandas                         as pd
import numpy                          as np
import sklearn.metrics                as metrics
import time
import copy

from sklearn.tree                   import DecisionTreeRegressor
from sklearn.ensemble               import RandomForestRegressor
from sklearn.linear_model           import LinearRegression
from sklearn.linear_model           import Ridge

from src.evaluate import evaluate_model, feature_importance
from src.models import run_lightgbm, run_models, ensemble_models, predict_ensemble, get_default_models

# Data Preparation

In [ ]:
# If running on Google Colab, uncomment below to mount Drive:
# from google.colab import drive
# drive.mount('/content/drive')
#
# train_url = r"/content/drive/MyDrive/preprocessed_dataset/train_dataset.csv"
# val_url = r"/content/drive/MyDrive/preprocessed_dataset/validation_dataset.csv"
# test_url = r"/content/drive/MyDrive/preprocessed_dataset/test_dataset.csv"

In [ ]:
# Locally
train_url = "../preprocessed_dataset/train_dataset.csv"
val_url = "../preprocessed_dataset/validation_dataset.csv"
test_url = "../preprocessed_dataset/test_dataset.csv"

In [17]:
# Load data
train_df = pd.read_csv(train_url)
val_df = pd.read_csv(val_url)

train_df.head()

,date,store_nbr,sales,onpromotion,oil_price,cluster,WageDay,year date,month date,day date,...,state_10,state_11,state_12,state_13,state_14,state_15,type_y_1,type_y_2,type_y_3,type_y_4
0,2013-01-01,9,0.0,0,93.13,6,0,2013,1,1,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2013-01-01,11,0.0,0,93.13,6,0,2013,1,1,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,2013-01-01,43,0.0,0,93.13,10,0,2013,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2013-01-01,37,0.0,0,93.13,2,0,2013,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,2013-01-01,33,0.0,0,93.13,3,0,2013,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [18]:
X_train = train_df.drop(["sales", "date"], axis=1).to_numpy()
y_train = train_df["sales"].to_numpy()

X_val = val_df.drop(["sales", "date"], axis=1).to_numpy()
y_val = val_df["sales"].to_numpy()

X_train.shape, y_train.shape

((1498228, 118), (1498228,))

In [ ]:
# feature_importance() imported from src.evaluate
feature_names = train_df.drop(["sales", "date"], axis=1).columns.tolist()
importance_df = feature_importance(X_train, y_train, feature_names, top_features=12)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Features"], importance_df["Mutual_Gain"], color="#4C72B0")
plt.xlabel("Mutual Information Gain")
plt.ylabel("Feature")
plt.title("Top Features by Mutual Information")
plt.tight_layout()
plt.savefig("../images/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# evaluate_model() and run_models() imported from src.evaluate and src.models
# Metrics: MAE, MSE, RMSE, MAX_ERROR, RMSLE (replaces MAPE), R2

In [ ]:
# run_lightgbm() imported from src.models

In [ ]:
models = get_default_models()

results = run_models(models, X_train, y_train, X_val, y_val)
results = pd.DataFrame(results)
results.T

# Making predictions using ensemble models

In [ ]:
# ensemble_models() and predict_ensemble() imported from src.models
# Uses copy.deepcopy to ensure each ensemble member is independently trained

In [ ]:
trained_ensemble = ensemble_models(DecisionTreeRegressor(), X_train, y_train, num_iters=5)
y_pred = predict_ensemble(trained_ensemble, X_val)
ensemble_results = evaluate_model(y_pred, y_val)
results["Ensemble (Decision Tree)"] = ensemble_results
results.T

In [ ]:
# Generate model comparison visualization for portfolio
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

model_names = results.columns.tolist()
mae_values = results.loc["MAE"].values
r2_values = results.loc["R2"].values

# Color: highlight best model
colors_mae = ["#DD5143" if v == min(mae_values) else "#4C72B0" for v in mae_values]
colors_r2 = ["#DD5143" if v == max(r2_values) else "#4C72B0" for v in r2_values]

# MAE chart (lower is better)
axes[0].barh(model_names, mae_values, color=colors_mae)
axes[0].set_xlabel("Mean Absolute Error (MAE)")
axes[0].set_title("MAE by Model (lower is better)")
axes[0].invert_yaxis()
for i, v in enumerate(mae_values):
    axes[0].text(v + 5, i, f"{v:.1f}", va="center", fontsize=9)

# R² chart (higher is better)
axes[1].barh(model_names, r2_values, color=colors_r2)
axes[1].set_xlabel("R² Score")
axes[1].set_title("R² by Model (higher is better)")
axes[1].set_xlim(0, 1)
axes[1].invert_yaxis()
for i, v in enumerate(r2_values):
    axes[1].text(v + 0.01, i, f"{v:.4f}", va="center", fontsize=9)

plt.suptitle("Model Comparison — Store Sales Forecasting", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../images/model_comparison.png", dpi=150, bbox_inches="tight")
plt.savefig("../images/thumbnail.png", dpi=150, bbox_inches="tight")
plt.show()

  # Predicting Test Data

 **We are going to use the best model from all, so we get the best performance**

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(n_estimators=30, max_depth=10, random_state=42),
    "Ridge": Ridge(),
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "LightGBM": "LightGBM",
    "Ensemble (Decision Tree)": "Ensemble Of Models"
}

In [28]:
best_model = results.T.sort_values(by="MAE", ascending=True).iloc[0].name
models[best_model]

DecisionTreeRegressor()

In [29]:
test_dataset = pd.read_csv(test_url)
test_dataset.drop(["date", "sales"], axis=1, inplace=True) # sales are coming from the alignment, but naturally doesn't give any informational gain.
test_dataset = test_dataset.to_numpy()

test_dataset.shape

(28512, 118)

# Predict on test data and post it in Kaggle

In [ ]:
# If running on Google Colab, upload your kaggle.json:
# from google.colab import files
# files.upload()

In [ ]:
test_preds = predict_ensemble(trained_ensemble, test_dataset)
len(test_preds)

In [33]:
test_preds = predict_ensemble_models(ensemble_of_models, test_dataset)
len(test_preds)

28512

In [34]:
id_column = pd.read_csv("sample_submission.csv")["id"]
id_column

0        3000888
1        3000889
2        3000890
3        3000891
4        3000892
          ...   
28507    3029395
28508    3029396
28509    3029397
28510    3029398
28511    3029399
Name: id, Length: 28512, dtype: int64

In [35]:
# Convert predictions into csv file
submission = pd.DataFrame({
    "id": id_column,
    "sales": test_preds
})

submission.to_csv("submission.csv", index=False)


In [36]:
submission.head()

,id,sales
0,3000888,5.0
1,3000889,0.0
2,3000890,7.0
3,3000891,2344.0
4,3000892,0.0


In [37]:
!kaggle competitions submit -c store-sales-time-series-forecasting -f submission.csv -m "Summer Project 2024 is Done"

100% 382k/382k [00:00<00:00, 557kB/s] 
Successfully submitted to Store Sales - Time Series Forecasting